In [2]:
%pip install transformers==4.41.0
%pip install pypdf pymupdf pdfplumber
%pip install langchain langchain_openai langchain_community
%pip install torch jsonformer torchvision transformers accelerate


^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install --upgrade jupyter ipywidgets widgetsnbextension jupyter nbextension enable --python widgetsnbextension

Note: you may need to restart the kernel to use updated packages.


ERROR: The --python option must be placed before the pip subcommand name


In [3]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader
loader1 = PyPDFLoader("2025_2.pdf")
# loader2 = PyMuPDFLoader("2025_2.pdf")
# loader3 = PDFPlumberLoader("2025_2.pdf")

In [4]:
docs1 = loader1.load()
# docs2 = loader2.load()
# docs3 = loader3.load()
print(f"Loaded {len(docs1)} documents with PyPDFLoader")
# print(f"Loaded {len(docs2)} documents with PyMuPDFLoader")
# print(f"Loaded {len(docs3)} documents with PDFPlumberLoader")

Loaded 308 documents with PyPDFLoader


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
load_dotenv()
from textwrap import dedent

분야_list = ["산업·중소기업·에너지", "공공질서및안전", "국토·교통", "기타"]
json_schema = {
    "type": "object",
    "properties": {
        "정책명": {
            "type": "string",
            "description": "정책의 이름"
        },
        "분야": {
            "type": "string",
            "enum": 분야_list,
            "description": "정책이 속하는 분야"
        },
        "세부분야": {
            "type": "string",
            "description": "정책의 세부 분야"
        },
        "시작일": {
            "type": "string",
            "format": "date",
            "description": "정책의 시작일"
        },
        "종료일": {
            "type": "string",
            "format": "date",
            "description": "정책의 종료일"
        },
        "세부내용": {
            "type": "string",
            "description": "정책의 전체 세부 내용"
        },
        "성공확률": {
            "type": "number",
            "description": "정책의 성공 확률 (0-100)"
        },
    "required": ["정책명", "분야", "세부분야", "시작일", "종료일", "세부내용", "성공확률"]
    }
}

model = ChatOpenAI(model="gpt-4o", temperature=0.2, max_tokens=1000)
prompt = ChatPromptTemplate.from_template(template=dedent("""
당신은 정책 요약 전문가입니다. 주어진 정책 문서에서 다음 정보를 추출하여 JSON 형식으로 반환하세요:
```
{{
"정책명": "정책의 이름",
"분야": "정책이 속하는 분야",
"세부분야": "정책의 세부 분야",
"시행일": "정책의 시작일",
"세부내용": "정책의 전체 세부 내용",
"기여도": "정책의 실제 사회 기여도 (0-100)"
}}
```
규칙:
1. 불필요한 특수문자나 공백, 불필요한 문자, "```json" 등의 문자는 절대 넣지 않습니다.
2. 만약, 주어진 문서가 특정 정책에 대한 내용이 아니라면 아무것도 출력하지 않습니다.
3. json 중 찾기 힘든 내용이라면 그 부분은 공백을 입력합니다.
4. "성공확률"은 주어진 정책 문서를 바탕으로 당신이 임의로 판단하는 값으로 비울 수 없습니다. 얼마나 사회에 기여할 수 있는지 0에서 100 사이의 값을 입력합니다.
5. "정책명", "분야", "세부내용"은 비울 수 없습니다.
6. "시행일"은 YYYY-MM-DD 형식으로 작성합니다.
7. "분야"는 "공공질서및안전" 으로 통일합니다.
주어진 정책 문서:
{input}
"""))


In [40]:
docs1[199].page_content

'176 1772025년 하반기부터 이렇게 달라집니다\n범정부 정보시스템 장애관리체계 강화\n☎ 044-205-2825행정안전부 디지털기반안전과\n행정정보시스템 장애로 인한 국민의 불편을 최소화하기 위해, 그간 기관별로 수행해왔던 \n장애관리를 범정부 차원에서 수행하여 보다 안정적으로 전자정부서비스를 제공합니다.\n    행정정보시스템의 장애를 예방하고 장애 발생 시 신속하게 대응·복구하기 위해 각 기관은 앞으로 \n행정안전부가 수립하는 범정부 차원의 지침에 따라 보다 체계적으로 계획을 수립하고 이행하게 \n됩니다. \n    이때, 전체 행정·공공정보 시스템은 중요도에 따라 등급이 나눠지고, 소관 기관은 이 등급에 \n따라 서비스 수준을 정하거나 투자우선순위를 정하여 보다 효율적으로 정보시스템을 관리하게 \n됩니다. \n    또한, 주요 정보시스템에서 장애가 발생하면, 국민이 서비스 중단·재개 상황과 대체 창구·수단을 \n알 수 있도록 즉각 공지하고 장애원인조사를 통해 유사한 장애가 재발하지 않도록 관리하게 됩니다.\n범정부 정보시스템 장애관리체계 강화\n추진배경 디지털행정서비스의 안정적인 제공을 위하여 범정부 정보시스템의 장애관리체계 강화\n주요내용  ·  (장애예방·대응) 장애관리를 위한 계획을 체계적으로 수립 ㆍ 시행\n ·  (등급 관리) 정보시스템의 중요도 ㆍ 영향도 등에 따라 체계적으로 정보시스템 관리\n ·  (현황조사 및 점검) 정보시스템의 안정성을 확인하기 위하여 현황조사 및 점검 수행\n ·  (장애상황 및 사후관리) 지체 없이 행정안전부에 장애상황 통보, 장애원인 조사·분석 \n수행\n시행일 2025년 7월 8일\n   ※ 전자정부법 제56조의2~5항, 전자정부법 시행령 제70조의2~5항\n행정안전부 www.mois.go.kr\n국가법령정보센터>법령명>전자정부법 국민참여입법센터>입법예고>전자정부법 시행령 일부개정령안 \n입법예고'

In [6]:
import os, json, time
os.makedirs("output", exist_ok=True)

# range_list = [range(95, 115), range(118, 132), range(193, 227)]
for i in range(193, 228):
    try:
        e = time.time()
        # json 형태 답변
        response = model.invoke(prompt.invoke({"input": docs1[i].page_content}))
        new_data_text = response.content.strip()
        # json 파싱
        try:
            new_data = json.loads(new_data_text)
        except json.JSONDecodeError as error:
            print("JSON 파싱 에러:", error)
            print("new_data_text : ", new_data_text)
            new_data = None
            continue
        if new_data is None:
            print("유효한 JSON 데이터가 아니므로 종료합니다.")
        else:
            filename = 'output/policy.json'

        # 기존 데이터 불러오기 (파일 없으면 빈 리스트로 시작)
        if os.path.exists(filename):
            with open(filename, 'r', encoding='utf-8') as f:
                try:
                    existing_data = json.load(f)
                except json.JSONDecodeError:
                    existing_data = []
        else:
            existing_data = []

        # existing_data가 리스트인지 확인. 아니면 리스트로 감싸기
        if not isinstance(existing_data, list):
            existing_data = [existing_data]

        # 같은 name 있는지 검사
        exists = any(item.get('정책명') == new_data.get('정책명') for item in existing_data)

        if exists:
            print(f"'{new_data.get('정책명')}' 정책이 이미 존재합니다. 추가하지 않습니다.")
        else:
            existing_data.append(new_data)
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(existing_data, f, ensure_ascii=False, indent=2)
            print(f"'{new_data.get('정책명')}' 정책 정보를 추가 저장했습니다.")

    except Exception as error:
        print(f"[{i}] 에러 발생:", error)
        continue
    finally:
        s = time.time()
        print(f"{i}번 째 정책 로딩에 {s-e} 초 걸렸습니다")

'아동학대범죄의 처벌 등에 관한 특례법 시행' 정책 정보를 추가 저장했습니다.
193번 째 정책 로딩에 6.8677146434783936 초 걸렸습니다
'외국인 기본인적정보 범정부 통일 기반 마련' 정책 정보를 추가 저장했습니다.
194번 째 정책 로딩에 4.3947367668151855 초 걸렸습니다
'범죄피해자의 재판기록 열람등사권 확대' 정책 정보를 추가 저장했습니다.
195번 째 정책 로딩에 4.08593487739563 초 걸렸습니다
196번 째 정책 로딩에 8.069279909133911 초 걸렸습니다


KeyboardInterrupt: 

In [26]:
print(response.content)

{
"정책명": "범죄피해자의 재판기록 열람등사권 확대",
"분야": "법무",
"세부분야": "형사법",
"시작일": "2025년 9월 19일",
"종료일": "",
"세부내용": "피해자의 알권리를 보장하고 재판 참여권을 실질적으로 보장하기 위해, 그간 제한적으로 인정되어 오던 범죄피해자의 재판기록 열람등사권을 확대하게 되었습니다. 형사재판이 계속 중인 사건의 피해자, 법정대리인, 피해자로부터 위임을 받은 배우자·직계친족·형제자매·변호사가 재판기록의 열람·등사를 신청하는 경우 재판장은 이를 원칙적으로 허가해야 합니다. 재판장이 예외적으로 허가하지 아니하거나 사용 목적의 제한 또는 조건을 붙여 허가하는 경우에는 열람·등사를 신청한 사람에게 그 이유를 통지하도록 하여 피해자의 알 권리를 보장하도록 하였습니다.",
"성공확률": ""
}


In [18]:
prompt

(ChatPromptTemplate(input_variables=['\n"정책명"'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['\n"정책명"'], input_types={}, partial_variables={}, template='\n당신은 정책 요약 전문가입니다. 주어진 정책 문서에서 다음 정보를 추출하여 JSON 형식으로 반환하세요:\n```\n{\n"정책명": "정책의 이름",\n"분야": "정책이 속하는 분야",\n"세부분야": "정책의 세부 분야",\n"시작일": "정책의 시작일",\n"종료일": "정책의 종료일",\n"세부내용": "정책의 전체 세부 내용",\n"성공확률": "정책의 성공 확률 (0-100)"\n}\n불필요한 특수문자나 공백, 불필요한 문자는 절대 넣지 않습니다.\n주어진 정책 문서:\n<bound method Kernel.raw_input of <ipykernel.ipkernel.IPythonKernel object at 0x0000027F64EFF800>>\n```\n'), additional_kwargs={})]),)

In [ ]:
print(doc)

170 1712025년 하반기부터 이렇게 달라집니다
아동학대범죄의 처벌 등에 관한 특례법 시행
☎ 02-2110-3648법무부 여성아동인권과
「아동학대범죄의 처벌 등에 관한 특례법」(2024. 12. 20. 개정) 개정법이 2025년 6월 21일부터 
시행됩니다.
※ 개정법에 신설한 아동학대살해미수죄 조문은 공포일인 2024년 12월 20일부터 시행
    학대피해아동이 친숙한 곳에서 보호받을 수 있도록 ‘피해아동 등을 연고자 등에게 인도’하는 
내용의 응급조치를 추가하였습니다.
    아동학대행위자에게 약식명령을 고지할 경우에도 아동학대 치료프로그램의 이수명령 병과가 
가능하도록 규정을 정비하였습니다.
    아동을 직접 교육·보호하는 현장에서 근무하는 대안교육기관 종사자에게도 아동학대 
신고의무를 부여하여 면밀한 아동보호가 이루어질 수 있도록 하였습니다.
    검사에게 임시조치 연장·취소·변경 청구권 및 피해아동보호명령 청구권을 부여하여 피해아동 
보호 공백을 방지하였습니다.
이번 개정으로 아동학대범죄에 엄정히 대응하고, 아동학대 사각지대를 해소하고 피해아동 
보호조치의 실효성을 강화함으로써 피해아동의 권익을 더욱 두텁게 보호할 것으로 기대됩니다.
「아동학대범죄 처벌 등에 관한 특례법」 시행
추진배경 아동학대 피해아동에 대한 보호조치 실효성을 강화함으로써 아동학대에 효과적으로 
대응할 수 있는 체계 구축 
주요내용 학대 피해아동을 연고자 등에게 인도하는 응급조치 및 약식명령과 이수명령 병과 근거 
신설, 아동 보호를 위한 검사의 청구권 확대 등
시행일 2025년 6월 21일
법무부 www.moj.go.kr
법무부 누리집>보도자료>“「아동학대범죄의 처벌 등에 관한 특례법」 개정안 국회 본회의 통과”


In [ ]:
# 기존 예시
import torch
vocab_size = model.config.vocab_size  # 확인 필요!
logits = torch.zeros((1, vocab_size))  # → 이게 문제가 됨
vocab_size, logits.shape

(51200, torch.Size([1, 51200]))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

# --------- 1. 데이터 예제 (세부내용은 긴 문장) ---------
data = [
    {
        "분야": "공공질서및안전",
        "세부분야": "CCTV설치",
        "시행부서": "도시안전과",
        "시작일": "2023-01-01",
        "종료일": "2023-12-31",
        "세부내용": """본 사업은 세종시 내 범죄 다발지역 50개소에 방범용 CCTV를 설치하여 범죄율을 낮추고 시민의 안전을 보장하기 위함입니다. 
        각 CCTV는 최신 AI 기반 범죄 탐지 기능을 갖추고 있으며 실시간 관제센터와 연동되어 빠른 대응이 가능하도록 설계되었습니다. 
        해당 사업은 시민 설문조사 결과를 반영하여 주요 민원 발생 지역 중심으로 추진되며, 설치 후 1년간 시범 운영과 성능 검증 절차를 거친 후 정식 운영될 예정입니다.""",
        "성공여부": 0.7
    },
    {
        "분야": "보건",
        "세부분야": "감염병대응",
        "시행부서": "보건정책과",
        "시작일": "2022-03-01",
        "종료일": "2023-08-15",
        "세부내용": """세종시 보건정책과는 감염병 확산 방지를 위해 전담 인력의 증원과 함께 시민 대상 예방 교육을 확대합니다. 
        본 정책은 신종 감염병 및 계절성 독감에 대비하여 의료기관과 연계된 협력체계를 강화하고, 선별진료소 운영 효율성을 높이기 위해 최신 장비를 도입합니다. 
        아울러, 감염병 취약계층 보호를 위해 지역사회 건강센터에서 예방접종 지원 및 방역물품 보급을 병행할 예정입니다.""",
        "성공여부": 0.6
    },
    {
    "분야": "환경",
    "세부분야": "생활폐기물관리",
    "시행부서": "환경관리과",
    "시작일": "2022-05-01",
    "종료일": "2023-04-30",
    "세부내용": """세종시 내 생활폐기물의 효율적 처리를 위해 RFID 기반 종량제를 확대 적용하고, 무단투기 감시를 위한 이동식 카메라를 도입합니다. 
    또한 주민참여형 분리배출 캠페인을 추진하며, 민간 수거업체와의 협력을 통해 청결도를 평가하고 인센티브를 제공합니다. 
    시민 교육과 홍보를 강화하여 쓰레기 감량 효과를 극대화할 계획입니다.""",
    "성공여부": 0.8
},
{
    "분야": "교통",
    "세부분야": "자전거도로확충",
    "시행부서": "교통정책과",
    "시작일": "2021-04-01",
    "종료일": "2022-12-31",
    "세부내용": """도심과 외곽을 연결하는 자전거 전용 도로 20km를 신규 조성하고, 주요 구간에는 자전거 무인 대여소와 쉼터를 설치합니다. 
    자전거 이용 활성화를 위해 공공기관 대상 출퇴근 장려 프로그램을 운영하고, 시민안전보험을 통해 사고에 대한 보장도 강화됩니다. 
    설문조사를 통해 우선 정비 구간을 선정하여 단계적으로 개선합니다.""",
    "성공여부": 0.4
},
{
    "분야": "복지",
    "세부분야": "육아지원서비스",
    "시행부서": "가족복지과",
    "시작일": "2023-01-15",
    "종료일": "2024-01-14",
    "세부내용": """영유아 부모를 위한 육아 지원 바우처를 도입하고, 지역 거점형 육아종합지원센터를 신설하여 일대일 맞춤 상담 및 프로그램을 제공합니다. 
    육아휴직 후 복귀자 대상 커리어 지원 서비스도 연계하며, 아빠 육아참여 활성화를 위한 공동육아 행사와 교육도 운영됩니다.""",
    "성공여부": 0.2
},
{
    "분야": "산업",
    "세부분야": "중소기업디지털전환",
    "시행부서": "경제정책과",
    "시작일": "2021-08-01",
    "종료일": "2022-12-31",
    "세부내용": """세종시 중소기업의 생산성 향상을 위해 디지털 전환 컨설팅을 무료로 제공하고, ERP 및 CRM 시스템 도입 비용의 일부를 지원합니다. 
    기술 세미나와 실무자 교육을 병행하여 인력 역량도 강화하며, 디지털 전환 성공 사례를 공유하는 온라인 플랫폼도 운영할 예정입니다.""",
    "성공여부": 0.3
},
{
    "분야": "도시계획",
    "세부분야": "노후주택정비",
    "시행부서": "도시재생과",
    "시작일": "2020-09-01",
    "종료일": "2022-06-30",
    "세부내용": """노후주택 밀집지역을 대상으로 안전진단을 실시하고, 긴급 보수가 필요한 가구에 대한 지원을 강화합니다. 
    자율주택 정비 사업을 통해 주민이 직접 사업을 기획·운영할 수 있도록 장려하며, 빈집을 활용한 공공시설 리모델링도 함께 추진됩니다. 
    주민설명회를 통해 의견을 수렴하고 사업 만족도를 지속적으로 평가합니다.""",
    "성공여부": 0.4
},
{
    "분야": "문화체육",
    "세부분야": "스포츠인프라확충",
    "시행부서": "체육진흥과",
    "시작일": "2022-03-01",
    "종료일": "2023-07-31",
    "세부내용": """생활체육 활성화를 위해 주민 이용이 가능한 실내체육관과 소규모 야외 운동시설을 조성합니다. 
    지역 동호회와 연계한 정기 프로그램을 운영하고, 청소년 대상 무료 스포츠 교실도 진행됩니다. 시설 예약 시스템을 전산화하여 접근성을 높이고, 
    안전한 이용을 위한 관리 인력도 추가 배치됩니다.""",
    "성공여부": 0.6
},
{
    "분야": "행정",
    "세부분야": "AI기반민원처리",
    "시행부서": "행정혁신과",
    "시작일": "2023-06-01",
    "종료일": "2024-05-31",
    "세부내용": """민원 응답 속도와 정확도 향상을 위해 AI 기반 챗봇을 도입하고, 민원 유형 분석을 통해 자동 분류 및 배분 시스템을 구축합니다. 
    다양한 언어와 장애인을 위한 보조 기능도 포함되며, 실시간 민원 처리 상황을 대시보드로 시각화하여 시민에게 공개합니다. 서비스 개선을 위한 피드백 시스템도 포함됩니다.""",
    "성공여부": 0.99
},
{
    "분야": "보건",
    "세부분야": "정신건강증진",
    "시행부서": "건강증진과",
    "시작일": "2021-02-01",
    "종료일": "2022-12-31",
    "세부내용": """정신건강복지센터를 통해 시민 대상 심리상담 서비스를 강화하고, 자살 예방 및 스트레스 완화 프로그램을 운영합니다. 
    지역 보건소와 협력하여 정신건강 교육을 확대하며, 정신과 전문의와 연계한 집중 상담 지원체계도 구축됩니다. 캠페인을 통해 정신건강에 대한 인식 제고 활동도 병행됩니다.""",
    "성공여부": 0.7
}
]


In [ ]:
# --------- 2. 범주형 변수 인코딩 ---------
le_field = LabelEncoder()
class le_subfield():
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
        self.bert_model = AutoModel.from_pretrained("bert-base-multilingual-cased")
    def fit_transform(self, text_list):
        tokenizer = self.tokenizer
        bert_model = self.bert_model
        with torch.no_grad():
            encoded = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
            model_output = bert_model(**encoded)
            # 문장 벡터는 CLS 토큰의 출력 사용
            embeddings = model_output.last_hidden_state[:, 0, :]
            return embeddings
    def transform(self, text_list):
        tokenizer = self.tokenizer
        bert_model = self.bert_model
        with torch.no_grad():
            encoded = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
            model_output = bert_model(**encoded)
            # 문장 벡터는 CLS 토큰의 출력 사용
            embeddings = model_output.last_hidden_state[:, 0, :]
            return embeddings
class le_dept():
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
        self.bert_model = AutoModel.from_pretrained("bert-base-multilingual-cased")
    def fit_transform(self, text_list):
        tokenizer = self.tokenizer
        bert_model = self.bert_model
        with torch.no_grad():
            encoded = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
            model_output = bert_model(**encoded)
            # 문장 벡터는 CLS 토큰의 출력 사용
            embeddings = model_output.last_hidden_state[:, 0, :]
            return embeddings
    def transform(self, text_list):
        tokenizer = self.tokenizer
        bert_model = self.bert_model
        with torch.no_grad():
            encoded = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
            model_output = bert_model(**encoded)
            # 문장 벡터는 CLS 토큰의 출력 사용
            embeddings = model_output.last_hidden_state[:, 0, :]
            return embeddings

fields = [d["분야"] for d in data]
subfields = [d["세부분야"] for d in data]
depts = [d["시행부서"] for d in data]

fields_enc = le_field.fit_transform(fields)
subfields_enc = le_subfield().fit_transform(subfields)
depts_enc = le_dept().fit_transform(depts)

fields_enc = torch.tensor(fields_enc, dtype=torch.float).unsqueeze(1)
# subfields_enc = torch.tensor(subfields_enc, dtype=torch.float).unsqueeze(1)
# depts_enc = torch.tensor(depts_enc, dtype=torch.float).unsqueeze(1)

# 날짜 처리: 총일수 계산
from datetime import datetime
dates = []
for d in data:
    start = datetime.strptime(d["시작일"], "%Y-%m-%d")
    end = datetime.strptime(d["종료일"], "%Y-%m-%d")
    total_days = (end - start).days
    dates.append([start.year, start.month, total_days])

# 스케일링
scaler = MinMaxScaler()
dates_scaled = scaler.fit_transform(dates)
dates_scaled = dates_scaled.mean(axis=1) 
dates_scaled = torch.tensor(dates_scaled, dtype=torch.float).unsqueeze(1)


# 라벨
labels = torch.tensor([d["성공여부"] for d in data], dtype=torch.float).unsqueeze(1)


In [ ]:
print(f"fields_enc: {fields_enc}, {fields_enc.shape}, {fields_enc.dtype}")
print(f"subfields_enc: {subfields_enc}, {subfields_enc.shape}, {subfields_enc.dtype}")
print(f"depts_enc: {depts_enc}, {depts_enc.shape}, {depts_enc.dtype}")
print(f"dates_scaled: {dates_scaled}, {dates_scaled.shape}, {dates_scaled.dtype}")
print(f"labels: {labels}, {labels.shape}, {labels.dtype}")

fields_enc: tensor([[0.],
        [4.],
        [8.],
        [1.],
        [5.],
        [6.],
        [2.],
        [3.],
        [7.],
        [4.]]), torch.Size([10, 1]), torch.float32
subfields_enc: tensor([[-0.0561, -0.0565,  0.2463,  ...,  0.2219,  0.1887, -0.2411],
        [-0.1840, -0.3941,  0.2361,  ...,  0.3724,  0.0748, -0.1403],
        [-0.1816, -0.0594,  0.4043,  ...,  0.0596,  0.1023,  0.2138],
        ...,
        [-0.0009, -0.0312,  0.0273,  ...,  0.3169,  0.2786, -0.1332],
        [-0.1626, -0.2187,  0.4910,  ...,  0.1755,  0.2918, -0.2151],
        [-0.1613, -0.3658,  0.0345,  ...,  0.3537,  0.2411, -0.0706]]), torch.Size([10, 768]), torch.float32
depts_enc: tensor([[-0.3460, -0.2878,  0.4879,  ...,  0.0358,  0.2312,  0.1364],
        [-0.2386, -0.2537,  0.4155,  ...,  0.1136,  0.2392,  0.3411],
        [-0.2777, -0.1273,  0.5739,  ..., -0.0197,  0.1237,  0.1690],
        ...,
        [-0.0853, -0.0426,  0.1823,  ...,  0.2661,  0.2322,  0.0253],
        [-0.2021, -0

In [ ]:
len(fields_enc)

10

In [ ]:
# --------- 3. 문장 임베딩 (사전학습 모델 사용) ---------
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/distiluse-base-multilingual-cased-v2")
bert_model = AutoModel.from_pretrained("sentence-transformers/distiluse-base-multilingual-cased-v2")

def embed_text(text_list):
    with torch.no_grad():
        encoded = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
        model_output = bert_model(**encoded)
        # 문장 벡터는 CLS 토큰의 출력 사용
        embeddings = model_output.last_hidden_state[:, 0, :]
        return embeddings

texts = [d["세부내용"] for d in data]
text_embeddings = embed_text(texts)

In [ ]:
print(f"text_embeddings.shape: {text_embeddings.shape}, {text_embeddings.dtype}")

text_embeddings.shape: torch.Size([10, 768]), torch.float32


In [ ]:
# --------- 4. PyTorch 모델 ---------
class PolicyEmbeddingModel(nn.Module):
    def __init__(self, field_dim, subfield_dim, dept_dim, text_dim, date_dim, hidden_dim=128):
        super(PolicyEmbeddingModel, self).__init__()
        self.field_fc = nn.Linear(field_dim, 8)          # One-hot 입력 처리
        # self.subfield_fc = nn.Linear(subfield_dim, 8)
        # self.dept_fc = nn.Linear(dept_dim, 8)
        self.fc_dates = nn.Linear(date_dim, 8)
        # self.text_fc = nn.Linear(text_dim, 8)  # 텍스트 임베딩 처리
        self.fc1 = nn.Linear(8+subfield_dim+dept_dim+8+text_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, field, subfield, dept, dates, text_emb):
        f = self.relu(self.field_fc(field))
        # sf = self.relu(self.subfield_fc(subfield))
        # d = self.relu(self.dept_fc(dept))
        dt = self.relu(self.fc_dates(dates))
        # print(f"f: {f.shape}, sf: {sf.shape}, d: {d.shape}, dt: {dt.shape}, text_emb: {text_emb.shape}")
        x = torch.cat([f, subfield, dept, dt, text_emb], dim=1)
        x = self.relu(self.fc1(x))
        out = self.sigmoid(self.fc2(x))
        return out


# 모델 초기화
model = PolicyEmbeddingModel(
    field_dim=fields_enc.shape[1],
    subfield_dim=subfields_enc.shape[1],
    dept_dim=depts_enc.shape[1],
    text_dim=text_embeddings.shape[1],
    date_dim=dates_scaled.shape[1]
)



In [ ]:
le_field.classes_

array(['공공질서및안전', '교통', '도시계획', '문화체육', '보건', '복지', '산업', '행정', '환경'],
      dtype='<U7')

In [ ]:
# --------- 5. 학습 ---------
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# # 입력 텐서로 변환
# field_tensor = torch.tensor(fields_enc, dtype=torch.float)
# subfield_tensor = torch.tensor(subfields_enc, dtype=torch.float)
# dept_tensor = torch.tensor(depts_enc, dtype=torch.float)
# date_tensor = torch.tensor(dates_scaled, dtype=torch.float)

# 학습 루프
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(fields_enc, subfields_enc, depts_enc, dates_scaled, text_embeddings)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss={loss.item():.4f}")


Epoch 0: Loss=0.6967
Epoch 10: Loss=0.5866
Epoch 20: Loss=0.5635
Epoch 30: Loss=0.5601
Epoch 40: Loss=0.5584
Epoch 50: Loss=0.5583
Epoch 60: Loss=0.5582
Epoch 70: Loss=0.5582
Epoch 80: Loss=0.5581
Epoch 90: Loss=0.5581


In [ ]:
# --------- 6. 신규 정책 예측 ---------
new_policy = {
    "분야": "보건",
    "세부분야": "감염병대응",  # ✅ 기존 데이터에 존재
    "시행부서": "보건정책과",  # ✅ 기존 데이터에 존재
    "시작일": "2025-05-01",
    "종료일": "2026-04-30",
    "세부내용": """
    세종시 보건정책과는 지역 내 감염병 확산을 예방하기 위해
    최신 방역 시스템 도입 및 대응 체계 구축을 추진합니다.
    본 사업은 의료기관 연계형 비상대응 훈련과 취약계층 대상
    백신 접종 지원을 포함하며, 재난대응 모바일 애플리케이션을 통해
    시민들이 감염병 정보를 신속하게 받아볼 수 있도록 할 예정입니다.
    """,
}
def test(new_policy):
    """
    하나의 신규 정책을 입력받아 성공 확률을 예측하는 함수입니다."""
    # 범주형 인코딩
    field_enc = torch.tensor([le_field.transform([new_policy["분야"]])[0]], dtype=torch.float).unsqueeze(1)
    subfield_enc = le_subfield().transform([new_policy["세부분야"]])
    dept_enc = le_dept().transform([new_policy["시행부서"]])
    # print(f"field_enc.shape: {field_enc.shape}, subfield_enc.shape: {subfield_enc.shape}, dept_enc.shape: {dept_enc.shape}")

    # 날짜 처리
    start = datetime.strptime(new_policy["시작일"], "%Y-%m-%d")
    end = datetime.strptime(new_policy["종료일"], "%Y-%m-%d")
    total_days = (end - start).days
    dates_scaled = scaler.transform([[start.year, start.month, total_days]])
    dates_scaled = torch.tensor(dates_scaled, dtype=torch.float)
    dates_scaled = dates_scaled.mean(axis=1)
    date_tensor_new = torch.tensor(dates_scaled, dtype=torch.float).unsqueeze(1)

    # 문장 임베딩
    text_emb_new = embed_text([new_policy["세부내용"]])

    # 예측
    with torch.no_grad():
        success_prob = model(field_enc, subfield_enc, dept_enc, date_tensor_new, text_emb_new).item()
        print(f"\n🚀 신규 정책 성공 확률: {success_prob*100:.2f}%")
test(new_policy)


🚀 신규 정책 성공 확률: 70.65%


C:\Users\jinhy\AppData\Local\Temp\ipykernel_265992\324261635.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  date_tensor_new = torch.tensor(dates_scaled, dtype=torch.float).unsqueeze(1)


In [ ]:
test_data = [
    {
        "분야": "공공질서및안전",
        "세부분야": "CCTV설치",
        "시행부서": "도시안전과",
        "시작일": "2023-01-01",
        "종료일": "2023-12-31",
        "세부내용": "본 사업은 세종시 내 범죄 다발지역 50개소에 방범용 CCTV를 설치하여 범죄율을 낮추고 시민의 안전을 보장하기 위함입니다. 각 CCTV는 최신 AI 기반 범죄 탐지 기능을 갖추고 있으며 실시간 관제센터와 연동되어 빠른 대응이 가능하도록 설계되었습니다. 해당 사업은 시민 설문조사 결과를 반영하여 주요 민원 발생 지역 중심으로 추진되며, 설치 후 1년간 시범 운영과 성능 검증 절차를 거친 후 정식 운영될 예정입니다.",
        "성공여부": 0.8
    },
    {
        "분야": "보건",
        "세부분야": "감염병대응",
        "시행부서": "보건정책과",
        "시작일": "2022-03-01",
        "종료일": "2023-08-15",
        "세부내용": "세종시 보건정책과는 감염병 확산 방지를 위해 전담 인력의 증원과 함께 시민 대상 예방 교육을 확대합니다. 본 정책은 신종 감염병 및 계절성 독감에 대비하여 의료기관과 연계된 협력체계를 강화하고, 선별진료소 운영 효율성을 높이기 위해 최신 장비를 도입합니다. 아울러, 감염병 취약계층 보호를 위해 지역사회 건강센터에서 예방접종 지원 및 방역물품 보급을 병행할 예정입니다.",
        "성공여부": 0.6
    },
    {
        "분야": "환경",
        "세부분야": "대기질개선",
        "시행부서": "환경보전과",
        "시작일": "2023-03-01",
        "종료일": "2024-02-28",
        "세부내용": "세종시는 미세먼지 저감을 위한 특별대책으로 대기질 모니터링 센서를 주요 도로 및 학교 주변에 설치하고, 실시간 정보를 시민에게 제공합니다. 또한, 노후 경유차 조기 폐차 지원 및 전기차 보급 확대를 위한 인센티브를 강화합니다. 친환경 대중교통 수단 도입도 병행하여 지속 가능한 환경 조성을 목표로 합니다.",
        "성공여부": 1
    },
    {
        "분야": "보건",
        "세부분야": "노인건강관리",
        "시행부서": "노인복지과",
        "시작일": "2022-07-01",
        "종료일": "2023-06-30",
        "세부내용": "노년층의 만성질환 예방을 위해 정기 건강검진과 방문 건강관리 서비스를 확대 운영합니다. 또한, 마을 단위 건강교실과 식생활 개선 교육을 통해 자가 건강관리 능력을 향상시키며, 고령자 중심의 맞춤형 운동 프로그램도 제공합니다.",
        "성공여부": 1
    },
    {
        "분야": "교통",
        "세부분야": "버스정보시스템고도화",
        "시행부서": "교통정보과",
        "시작일": "2023-01-01",
        "종료일": "2023-12-31",
        "세부내용": "세종시는 시민의 교통 편의를 위해 실시간 버스 도착 정보를 제공하는 BIS를 고도화합니다. 정류장에 전자안내판을 확대 설치하고, 모바일 앱을 통해 위치 기반 예측 서비스를 제공합니다. 교통 데이터 분석을 통해 배차 효율성도 개선합니다.",
        "성공여부": 1
    },
    {
        "분야": "문화체육",
        "세부분야": "지역문화축제지원",
        "시행부서": "문화예술과",
        "시작일": "2022-04-01",
        "종료일": "2022-10-31",
        "세부내용": "지역 특색을 살린 문화 축제를 연중 개최하여 주민 참여를 촉진하고, 청년 예술가와 전통 문화 보존 단체의 협업을 통해 창의적인 프로그램을 운영합니다. 외부 관광객 유입을 위한 홍보 마케팅도 병행하여 지역 경제 활성화에 기여합니다.",
        "성공여부": 0
    },
    {
        "분야": "도시계획",
        "세부분야": "스마트시티인프라",
        "시행부서": "도시계획과",
        "시작일": "2021-06-01",
        "종료일": "2023-12-31",
        "세부내용": "세종시의 스마트시티 조성을 위해 공공 와이파이 확대, 스마트 가로등 설치, 통합관제센터 고도화 등 IT 인프라 확장을 추진합니다. 시민 참여형 플랫폼을 통해 교통, 환경, 안전 데이터를 통합 관리하여 효율적인 도시 운영을 목표로 합니다.",
        "성공여부": 1
    },
    {
        "분야": "산업",
        "세부분야": "지역특화산업육성",
        "시행부서": "산업진흥과",
        "시작일": "2022-01-01",
        "종료일": "2023-12-31",
        "세부내용": "세종형 특화산업 육성을 위해 바이오헬스 및 스마트농업 분야의 창업기업에 대한 맞춤형 지원을 제공합니다. 기술 개발, 판로 개척, 인재 양성을 통합 지원하며, 산학연 협력을 통해 지속 가능한 산업 생태계를 구축합니다.",
        "성공여부": 1
    },
    {
        "분야": "복지",
        "세부분야": "청년자립지원",
        "시행부서": "청년정책과",
        "시작일": "2023-05-01",
        "종료일": "2024-04-30",
        "세부내용": "청년층의 자립 기반 마련을 위해 주거 지원, 직업 훈련, 창업 멘토링을 통합 제공하는 청년지원센터를 운영합니다. 특히, 주거비 부담 완화를 위한 청년 월세 지원 사업을 확대하고, 맞춤형 진로 컨설팅을 정기적으로 실시합니다.",
        "성공여부": 1
    },
    {
        "분야": "행정",
        "세부분야": "민원통합처리시스템",
        "시행부서": "정보통신과",
        "시작일": "2022-09-01",
        "종료일": "2023-12-31",
        "세부내용": "각 부서별로 분산되어 있던 민원 접수 및 처리 시스템을 하나의 플랫폼으로 통합하여, 민원인의 불편을 해소합니다. 통합 시스템은 AI 기반 분류 기능을 탑재해 민원 유형에 따른 자동 배정이 가능하며, 처리 결과 알림도 실시간으로 제공됩니다.",
        "성공여부": 0
    },
    {
        "분야": "공공질서및안전",
        "세부분야": "재난대응훈련",
        "시행부서": "재난관리과",
        "시작일": "2023-02-01",
        "종료일": "2023-11-30",
        "세부내용": "지진, 화재, 감염병 등 다양한 재난 유형에 대비한 민·관 합동 대응 훈련을 정례화합니다. 지역별 위험요소를 분석한 시나리오를 바탕으로 실전 중심의 훈련을 진행하고, 시민 대상 재난 대처 교육과 훈련 참여를 의무화합니다.",
        "성공여부": 1
    },
    {
        "분야": "환경",
        "세부분야": "도시숲조성",
        "시행부서": "산림녹지과",
        "시작일": "2022-03-01",
        "종료일": "2023-10-31",
        "세부내용": "도심 내 유휴지를 활용한 도시숲을 조성하여 미세먼지 저감과 힐링 공간을 제공합니다. 주민참여형 나무심기 행사와 학교숲 조성도 병행하며, 생물 다양성 확보를 위한 식재 기준도 적용됩니다.",
        "성공여부": 1
    },
    {
        "분야": "산업",
        "세부분야": "청년창업지원",
        "시행부서": "창업지원과",
        "시작일": "2023-04-01",
        "종료일": "2024-03-31",
        "세부내용": "청년 예비창업자를 대상으로 창업 공간 제공, 시제품 제작비 지원, 창업 교육 및 투자 연계를 통합적으로 지원합니다. 또한 창업 성과를 주기적으로 평가하고 후속 지원 여부를 결정하는 체계도 마련됩니다.",
        "성공여부": 1
    },
    {
        "분야": "보건",
        "세부분야": "비만예방캠페인",
        "시행부서": "건강증진과",
        "시작일": "2022-06-01",
        "종료일": "2023-05-31",
        "세부내용": "비만율 증가에 대응하여 전 연령층을 대상으로 맞춤형 건강생활 캠페인을 전개합니다. 학교 급식 개선, 직장인 건강관리 프로그램, 대국민 건강 걷기 챌린지 등을 통해 체중 관리에 대한 인식을 개선하고, 참여 유도를 위한 모바일 앱도 개발됩니다.",
        "성공여부": 0
    },
    {
        "분야": "문화체육",
        "세부분야": "생활체육활성화",
        "시행부서": "체육지원과",
        "시작일": "2023-01-01",
        "종료일": "2023-12-31",
        "세부내용": "생활체육 참여율을 높이기 위해 공공 체육시설 이용 요금을 인하하고, 다양한 연령대별 운동 프로그램을 운영합니다. 지역 동호회 연계 지원과 함께 비대면 운동 콘텐츠도 제작하여 접근성을 강화합니다.",
        "성공여부": 1
    },
    {
        "분야": "행정",
        "세부분야": "전자문서시스템도입",
        "시행부서": "행정혁신과",
        "시작일": "2022-01-01",
        "종료일": "2022-12-31",
        "세부내용": "문서 행정의 디지털화를 위해 전자문서시스템을 도입하고, 부서 간 협업 문서 공유 체계를 강화합니다. 전자결재 시스템을 연계하여 업무 효율성을 높이며, 사용자 교육을 통해 활용도를 높입니다.",
        "성공여부": 1
    },
    {
        "분야": "복지",
        "세부분야": "긴급복지지원",
        "시행부서": "복지정책과",
        "시작일": "2021-10-01",
        "종료일": "2022-09-30",
        "세부내용": "실직, 질병, 재해 등 긴급 상황에 처한 저소득 가구를 대상으로 단기 생계비와 의료비를 지원하는 긴급복지지원제도를 확대 운영합니다. 선제적 발굴을 위한 상담 인력을 확충하고, 민간단체와의 협력 네트워크도 강화합니다.",
        "성공여부": 0
    }
]

In [ ]:
a = [{
        "분야": "교육",
        "세부분야": "디지털교육확산",
        "시행부서": "교육지원과",
        "시작일": "2021-03-01",
        "종료일": "2022-12-31",
        "세부내용": "관내 초중고등학교에 스마트 패드를 보급하고, 디지털 교과 콘텐츠와 연계한 맞춤형 학습 시스템을 구축합니다. 교사 대상 디지털 역량 강화 연수를 병행하고 있으며, 학부모 대상 설명회도 정기적으로 개최하여 교육의 신뢰도를 높이고자 합니다.",
        "성공여부": 0.9
    },
    {
        "분야": "환경",
        "세부분야": "미세먼지저감",
        "시행부서": "환경정책과",
        "시작일": "2020-01-01",
        "종료일": "2020-12-31",
        "세부내용": "노후 경유차 조기 폐차 지원과 함께 공공기관 차량을 친환경차로 교체하며, 대기질 측정소를 확대 설치하여 실시간 데이터를 공개합니다. 시민 자율 감시단을 조직하여 환경 보호 활동에 직접 참여할 수 있도록 유도합니다.",
        "성공여부": 0.7
    },
    {
        "분야": "교통",
        "세부분야": "대중교통개선",
        "시행부서": "교통과",
        "시작일": "2023-04-01",
        "종료일": "2024-03-31",
        "세부내용": "버스 정류장 정보 시스템 고도화와 저상버스 도입 확대를 추진합니다. 또한 배차 간격 단축과 정시성 향상을 위해 노선 조정을 실시하며, 시민 피드백 기반의 실시간 교통정보 앱을 개발 중입니다.",
        "성공여부": 0.85
    },
    {
        "분야": "복지",
        "세부분야": "노인복지강화",
        "시행부서": "복지지원과",
        "시작일": "2021-01-01",
        "종료일": "2022-12-31",
        "세부내용": "노인복지관을 신설하고, 건강검진 및 여가 프로그램을 확대합니다. 복지 사각지대에 놓인 독거노인 대상 방문간호 서비스도 신규 도입되며, 노인 일자리 창출과 연계한 기업 협력 프로젝트도 포함됩니다.",
        "성공여부": 0.75
    },
    {
        "분야": "문화체육",
        "세부분야": "문화예술교육",
        "시행부서": "문화정책과",
        "시작일": "2022-06-01",
        "종료일": "2023-06-30",
        "세부내용": "지역 예술인과 협업하여 초중고 대상 문화예술 교육 프로그램을 운영하고, 방과후 학교와 연계된 창작 워크숍도 정기 개최됩니다. 학교 외 교육시설 활용률을 높이기 위한 정책적 시도가 포함되어 있습니다.",
        "성공여부": 0.65
    },
    {
        "분야": "산업",
        "세부분야": "청년창업지원",
        "시행부서": "일자리정책과",
        "시작일": "2021-09-01",
        "종료일": "2023-02-28",
        "세부내용": "청년 창업자를 위한 창업보육센터를 확충하고, 멘토링 및 시제품 제작 비용을 지원합니다. 선정된 팀은 IR 피칭 기회를 통해 민간 투자 유치도 도모할 수 있도록 프로그램이 설계되었습니다.",
        "성공여부": 0.88
    },
    {
        "분야": "도시계획",
        "세부분야": "스마트시티구축",
        "시행부서": "도시계획과",
        "시작일": "2020-05-01",
        "종료일": "2022-10-30",
        "세부내용": "공공 와이파이 확대, 지능형 교통 시스템 도입, 환경센서 네트워크를 기반으로 한 데이터 기반 도시관리 체계 도입을 골자로 합니다. 스마트 가로등과 CCTV를 통합 운영하여 에너지 효율을 높입니다.",
        "성공여부": 0.92
    },
    {
        "분야": "행정",
        "세부분야": "민원행정개선",
        "시행부서": "민원서비스과",
        "시작일": "2023-01-01",
        "종료일": "2023-12-31",
        "세부내용": "온라인 민원처리 포털을 개선하고, 민원 자동 응답 시스템을 도입합니다. 다양한 언어 지원 기능을 추가하며, 민원 응답 속도 향상을 위한 내부 프로세스도 함께 개편 중입니다.",
        "성공여부": 0.78
    }
]


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
for new_data in test_data:
    try:
        test(new_data)
        print(f"✅ Successfully processed {new_data['분야']} - {new_data['세부분야']}")
    except Exception as e:
        print(f"Error processing {new_data['분야']} - {new_data['세부분야']}: {e}")
        continue


🚀 신규 정책 성공 확률: 69.96%
✅ Successfully processed 공공질서및안전 - CCTV설치

🚀 신규 정책 성공 확률: 59.94%
✅ Successfully processed 보건 - 감염병대응

🚀 신규 정책 성공 확률: 75.27%
✅ Successfully processed 환경 - 대기질개선

🚀 신규 정책 성공 확률: 48.91%
✅ Successfully processed 보건 - 노인건강관리

🚀 신규 정책 성공 확률: 68.53%
✅ Successfully processed 교통 - 버스정보시스템고도화

🚀 신규 정책 성공 확률: 54.47%
✅ Successfully processed 문화체육 - 지역문화축제지원

🚀 신규 정책 성공 확률: 68.60%
✅ Successfully processed 도시계획 - 스마트시티인프라

🚀 신규 정책 성공 확률: 52.99%
✅ Successfully processed 산업 - 지역특화산업육성

🚀 신규 정책 성공 확률: 44.05%
✅ Successfully processed 복지 - 청년자립지원

🚀 신규 정책 성공 확률: 92.63%
✅ Successfully processed 행정 - 민원통합처리시스템

🚀 신규 정책 성공 확률: 72.67%
✅ Successfully processed 공공질서및안전 - 재난대응훈련

🚀 신규 정책 성공 확률: 42.37%
✅ Successfully processed 환경 - 도시숲조성

🚀 신규 정책 성공 확률: 58.01%
✅ Successfully processed 산업 - 청년창업지원

🚀 신규 정책 성공 확률: 70.66%
✅ Successfully processed 보건 - 비만예방캠페인

🚀 신규 정책 성공 확률: 39.00%
✅ Successfully processed 문화체육 - 생활체육활성화

🚀 신규 정책 성공 확률: 77.61%
✅ Successfully processed 행정 - 전자문서시스템도입

🚀 신규 정책 